# 05 — Income Correlation: Who Benefits Most?

Explores whether higher-income counties tend to have lower assessment ratios
(i.e., receive more Prop 13 benefit). High-appreciation areas — often wealthier coastal counties —
have seen the largest divergence between assessed and market values over time.

This analysis tests the hypothesis that Prop 13 disproportionately benefits
higher-income homeowners in high-appreciation regions.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

ROOT = Path().resolve().parent
df = pd.read_csv(ROOT / 'data' / 'processed' / 'county_prop13.csv')
df = df.dropna(subset=['assessment_ratio', 'median_income', 'tax_gap_per_household'])
print(f'Loaded {len(df)} counties for correlation analysis')

Loaded 58 counties for correlation analysis


In [2]:
# Scatter: median income vs assessment ratio
slope, intercept, r, p, se = stats.linregress(
    df['median_income'], df['assessment_ratio']
)
r2 = r ** 2
print(f'Pearson r = {r:.3f}, R² = {r2:.3f}, p = {p:.4f}')

df['size_proxy'] = (df['tax_gap_annual_millions'] / df['tax_gap_annual_millions'].max() * 50 + 8)

fig1 = px.scatter(
    df,
    x='median_income',
    y='assessment_ratio',
    text='county',
    size='tax_gap_annual_millions',
    color='assessment_ratio',
    color_continuous_scale='RdYlGn',
    range_color=[0.3, 1.0],
    title=f'Median Household Income vs. Assessment Ratio by County<br>'
          f'<sup>R² = {r2:.2f} — bubble size = total annual tax gap | lower ratio = more Prop 13 benefit</sup>',
    labels={
        'median_income': 'Median Household Income ($)',
        'assessment_ratio': 'Assessment Ratio (AV / Market Value)',
        'tax_gap_annual_millions': 'Annual Tax Gap ($M)',
    },
    hover_data={
        'median_income': ':$,.0f',
        'assessment_ratio': ':.2%',
        'tax_gap_per_household': ':$,.0f',
        'tax_gap_annual_millions': ':$,.0f',
    },
)
# Add regression line
x_range = np.linspace(df['median_income'].min(), df['median_income'].max(), 100)
y_range = intercept + slope * x_range
fig1.add_trace(go.Scatter(
    x=x_range, y=y_range,
    mode='lines',
    line=dict(color='#555', dash='dash', width=1.5),
    name=f'Trend (r={r:.2f})',
    showlegend=True,
))
fig1.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig1.update_layout(
    height=600,
    coloraxis_showscale=False,
    xaxis=dict(tickprefix='$', tickformat=',.0f'),
    yaxis=dict(tickformat='.0%'),
)
fig1.show()

Pearson r = -0.426, R² = 0.181, p = 0.0009


In [3]:
# Scatter: median income vs tax gap per household
slope2, intercept2, r2val, p2, se2 = stats.linregress(
    df['median_income'], df['tax_gap_per_household']
)
r2_2 = r2val ** 2
print(f'Income vs. gap/HH: Pearson r = {r2val:.3f}, R² = {r2_2:.3f}, p = {p2:.4f}')

fig2 = px.scatter(
    df,
    x='median_income',
    y='tax_gap_per_household',
    text='county',
    color='median_income',
    color_continuous_scale='Viridis',
    title=f'Median Income vs. Annual Tax Gap per Household<br>'
          f'<sup>R² = {r2_2:.2f} — higher income counties tend to receive larger per-household Prop 13 benefit</sup>',
    labels={
        'median_income': 'Median Household Income ($)',
        'tax_gap_per_household': 'Annual Tax Gap per Household ($)',
    },
    hover_data={
        'median_income': ':$,.0f',
        'tax_gap_per_household': ':$,.0f',
        'assessment_ratio': ':.2%',
    },
)
x_range2 = np.linspace(df['median_income'].min(), df['median_income'].max(), 100)
y_range2 = intercept2 + slope2 * x_range2
fig2.add_trace(go.Scatter(
    x=x_range2, y=y_range2,
    mode='lines',
    line=dict(color='#e74c3c', dash='dash', width=1.5),
    name=f'Trend (r={r2val:.2f})',
    showlegend=True,
))
fig2.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig2.update_layout(
    height=600,
    coloraxis_showscale=False,
    xaxis=dict(tickprefix='$', tickformat=',.0f'),
    yaxis=dict(tickprefix='$', tickformat=',.0f'),
)
fig2.show()

Income vs. gap/HH: Pearson r = 0.683, R² = 0.467, p = 0.0000


In [4]:
# Summary: quintile breakdown
df['income_quintile'] = pd.qcut(df['median_income'], 5, labels=[
    'Lowest 20%', '20-40%', '40-60%', '60-80%', 'Highest 20%'
])
qdf = df.groupby('income_quintile', observed=True).agg(
    avg_assessment_ratio=('assessment_ratio', 'mean'),
    avg_gap_per_hh=('tax_gap_per_household', 'mean'),
    total_gap_millions=('tax_gap_annual_millions', 'sum'),
    n_counties=('county', 'count'),
).reset_index()

fig3 = px.bar(
    qdf,
    x='income_quintile',
    y='avg_gap_per_hh',
    color='avg_gap_per_hh',
    color_continuous_scale='Reds',
    title='Average Annual Tax Gap per Household by County Income Quintile<br>'
          '<sup>Higher-income counties receive systematically larger Prop 13 benefits</sup>',
    labels={
        'income_quintile': 'County Income Quintile',
        'avg_gap_per_hh': 'Avg Annual Tax Gap per HH ($)',
    },
    text='avg_gap_per_hh',
)
fig3.update_traces(texttemplate='$%{y:,.0f}', textposition='outside')
fig3.update_layout(
    coloraxis_showscale=False,
    height=420,
    yaxis=dict(tickprefix='$', tickformat=',.0f'),
)
fig3.show()

print(qdf.to_string(index=False))

income_quintile  avg_assessment_ratio  avg_gap_per_hh  total_gap_millions  n_counties
     Lowest 20%              1.000000        0.000000            0.000000          12
         20-40%              0.924031      318.356333          454.186641          11
         40-60%              0.928960      430.545009         2558.395367          12
         60-80%              0.871329      987.343169         2096.825396          11
    Highest 20%              0.869532     1788.171444         6039.880181          12
